In [ ]:
# Встановлюємо numpy версії 1.x, щоб уникнути конфлікту з scipy/albumentations
!pip install -U "numpy<2.0" scipy albumentations ultralytics opencv-python-headless

In [8]:
import ultralytics
import albumentations as A
import cv2
import numpy as np
import os
import shutil
import yaml
from tqdm import tqdm

print(f"Setup complete. YOLO version: {ultralytics.__version__}")


# Input paths
INPUT_TRAIN_DIR = "/kaggle/input/mydatasetmc/Mcdonalds.v1-mcdonald.yolov11/train"
INPUT_VALID_DIR = "/kaggle/input/mydatasetmc/Mcdonalds.v1-mcdonald.yolov11/valid"

# Output paths
OUTPUT_BASE = "/kaggle/working/dataset"
OUT_TRAIN_IMG = os.path.join(OUTPUT_BASE, "train/images")
OUT_TRAIN_LBL = os.path.join(OUTPUT_BASE, "train/labels")
OUT_VALID_IMG = os.path.join(OUTPUT_BASE, "valid/images")
OUT_VALID_LBL = os.path.join(OUTPUT_BASE, "valid/labels")

# Creating folders
for p in [OUT_TRAIN_IMG, OUT_TRAIN_LBL, OUT_VALID_IMG, OUT_VALID_LBL]:
    os.makedirs(p, exist_ok=True)

# AUGMENTATION 
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.Blur(blur_limit=5, p=0.3),
    A.RandomGamma(p=0.5),
    A.ColorJitter(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.7)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

print("Starting augmentation of training data...")

# Iterate over TRAIN images
img_names = [f for f in os.listdir(os.path.join(INPUT_TRAIN_DIR, "images")) if f.endswith(('.jpg', '.png', '.jpeg'))]

for fname in tqdm(img_names):
    img_path = os.path.join(INPUT_TRAIN_DIR, "images", fname)
    lbl_path = os.path.join(INPUT_TRAIN_DIR, "labels", fname.rsplit('.', 1)[0] + ".txt")

    # Read image
    image = cv2.imread(img_path)
    if image is None:
        continue
    
    # Read annotations (if label file missing → keep empty)
    bboxes = []
    class_labels = []
    
    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            for line in f:
                parts = list(map(float, line.split()))
                if len(parts) >= 5:
                    class_labels.append(int(parts[0]))
                    bboxes.append(parts[1:])  # x, y, w, h
    
    # 1. Save original image and label
    cv2.imwrite(os.path.join(OUT_TRAIN_IMG, fname), image)
    if os.path.exists(lbl_path):
        shutil.copy(lbl_path, os.path.join(OUT_TRAIN_LBL, fname.rsplit('.', 1)[0] + ".txt"))

    # 2. Create 5 augmented versions
    try:
        for i in range(5):
            aug = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            
            save_name = f"{fname.rsplit('.', 1)[0]}_aug{i}"
            
            # Save augmented image
            cv2.imwrite(os.path.join(OUT_TRAIN_IMG, save_name + ".jpg"), aug['image'])
            
            # Save augmented labels if exist
            if aug['bboxes']:
                with open(os.path.join(OUT_TRAIN_LBL, save_name + ".txt"), "w") as f:
                    for lbl, (x, y, w, h) in zip(aug['class_labels'], aug['bboxes']):
                        x, y, w, h = [min(max(v, 0.0), 1.0) for v in [x, y, w, h]]
                        f.write(f"{lbl} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")
    except Exception as e:
        print(f"Error augmenting {fname}: {e}")

print("Copying validation dataset...")

# Copy validation files without modifications
for folder in ["images", "labels"]:
    src = os.path.join(INPUT_VALID_DIR, folder)
    dst = os.path.join(OUTPUT_BASE, "valid", folder)
    if os.path.exists(src):
        for f in os.listdir(src):
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))

print("Dataset is ready.")

import yaml
import numpy as np
import ultralytics
from ultralytics import YOLO

# Version check
print(f"Current NumPy version: {np.__version__}")

if np.__version__.startswith("2"):
    raise RuntimeError("NumPy 2.x detected. Restart the Kaggle session and try again.")
else:
    print("NumPy 1.x detected. Training can proceed.")

# Restore paths
OUTPUT_BASE = "/kaggle/working/dataset"
OUT_TRAIN_IMG = os.path.join(OUTPUT_BASE, "train/images")
OUT_VALID_IMG = os.path.join(OUTPUT_BASE, "valid/images")
yaml_path = "/kaggle/working/data.yaml"

# Recreate YAML file if missing
if not os.path.exists(yaml_path):
    yaml_content = {
        'train': os.path.abspath(OUT_TRAIN_IMG),
        'val': os.path.abspath(OUT_VALID_IMG),
        'nc': 1,
        'names': ['McDonalds']
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)

print("Starting training...")
model = YOLO("yolo11n.pt")

results = model.train(
    data=yaml_path,
    epochs=15,
    imgsz=640,
    batch=16,
    name='mcdonalds_detect',
    device=[0],
    exist_ok=True,
    plots=False
)

/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Setup complete. YOLO version: 8.3.233
Starting augmentation of training data...


100%|██████████| 130/130 [00:07<00:00, 16.70it/s]


Copying validation dataset...
Dataset is ready.
Current NumPy version: 1.26.4
NumPy 1.x detected. Training can proceed.
Starting training...
Ultralytics 8.3.233 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=Fals

In [9]:
import glob
import cv2
from ultralytics import YOLO
import os

# --- 1. Search for model weights ---
weights_list = glob.glob("/kaggle/working/runs/detect/mcdonalds_detect*/weights/best.pt")

if not weights_list:
    print("Error: best.pt weights file not found.")
    exit()

best_weights = weights_list[0]
print(f"Found weights: {best_weights}")

# Paths
video_path = "/kaggle/input/videomc/McDonalds Delivery.mp4"
temp_output = "/kaggle/working/temp_video.mp4"   
final_output = "/kaggle/working/result_final.mp4"  

# Model initialization 
try:
    model = YOLO(best_weights)
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

# Video processing 
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Error: Cannot open video file {video_path}")
else:
    # Video parameters
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Use 'mp4v' for temporary encoding
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(temp_output, fourcc, fps, (w, h))

    print(f"Starting processing of {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames...")
    
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Detection
        results = model(frame, conf=0.9, verbose=False)
        annotated = results[0].plot()
        
        # Write the processed frame
        out.write(annotated)
        
        frame_count += 1
        if frame_count % 50 == 0:
            print(f"Processed {frame_count} frames")

    # Important: release file handles
    cap.release()
    out.release()
    print("Initial video processing completed.")

    # Convert mp4v → h264
    os.system(
        f'ffmpeg -y -i "{temp_output}" -vcodec libx264 -pix_fmt yuv420p "{final_output}" '
        f'-hide_banner -loglevel error'
    )
    
    # Remove temporary file
    if os.path.exists(temp_output):
        os.remove(temp_output)

    print("Done. The final result is saved as result_final.mp4")

Found weights: /kaggle/working/runs/detect/mcdonalds_detect/weights/best.pt
Starting processing of 498 frames...
Processed 50 frames
Processed 100 frames
Processed 150 frames
Processed 200 frames
Processed 250 frames
Processed 300 frames
Processed 350 frames
Processed 400 frames
Processed 450 frames
Initial video processing completed.
Done. The final result is saved as result_final.mp4
